In [ ]:
#BiLSTM based QC
import os
import pickle
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns

from gensim.models import KeyedVectors
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    precision_recall_fscore_support
)

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Bidirectional, LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

# ======================================================
# 1. Configuration
# ======================================================
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

DATA_PATH = "AmHQA.csv"
WORD2VEC_PATH = "amharic_word2vec.bin"

QUESTION_COL = "Question"
LABEL_COL = "Category"

MAX_WORDS = 30000
MAX_LEN = 50
BATCH_SIZE = 32
EPOCHS = 30

# Keep pretrained Word2Vec embeddings fixed initially.
# Set True to fine-tune embeddings during training.
EMBEDDING_TRAINABLE = False

# ======================================================
# 2. Load and clean the dataset
# ======================================================
df = pd.read_csv(DATA_PATH, encoding="utf-8")

required_cols = [QUESTION_COL, LABEL_COL]
missing_cols = [c for c in required_cols if c not in df.columns]

if missing_cols:
    raise ValueError(
        f"Missing columns: {missing_cols}. "
        f"Available columns: {df.columns.tolist()}"
    )

df = df[required_cols].dropna().copy()
df[QUESTION_COL] = df[QUESTION_COL].astype(str).str.strip()
df[LABEL_COL] = df[LABEL_COL].astype(str).str.strip()

df = df[
    (df[QUESTION_COL] != "") &
    (df[LABEL_COL] != "")
].drop_duplicates(subset=[QUESTION_COL]).reset_index(drop=True)

print("Dataset size:", df.shape)
print("\nClass distribution:")
print(df[LABEL_COL].value_counts())

# ======================================================
# 3. Encode question categories
# ======================================================
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(df[LABEL_COL])

num_classes = len(label_encoder.classes_)

print("\nQuestion categories:", list(label_encoder.classes_))
print("Number of classes:", num_classes)

if num_classes < 2:
    raise ValueError("At least two question categories are required.")

# ======================================================
# 4. Stratified train/validation/test split
#    80% train, 10% validation, 10% test
# ======================================================
X_train, X_temp, y_train, y_temp = train_test_split(
    df[QUESTION_COL].tolist(),
    y,
    test_size=0.20,
    random_state=SEED,
    stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=SEED,
    stratify=y_temp
)

print(
    f"\nTrain: {len(X_train)} "
    f"| Validation: {len(X_val)} "
    f"| Test: {len(X_test)}"
)

# ======================================================
# 5. Load pretrained Word2Vec vectors
# ======================================================
if not os.path.exists(WORD2VEC_PATH):
    raise FileNotFoundError(
        f"File not found: {WORD2VEC_PATH}. "
        "Update WORD2VEC_PATH to the actual file location."
    )

# Standard word2vec binary format:
word_vectors = KeyedVectors.load_word2vec_format(
    WORD2VEC_PATH,
    binary=True
)

embedding_dim = word_vectors.vector_size

print("\nWord2Vec vocabulary:", len(word_vectors.key_to_index))
print("Embedding dimension:", embedding_dim)

# If the file was saved using Word2Vec.save() instead,
# use the following two lines instead of the loader above:
#
# from gensim.models import Word2Vec
# word_vectors = Word2Vec.load(WORD2VEC_PATH).wv

# ======================================================
# 6. Tokenize questions
#    Fit tokenizer on training data only
# ======================================================
tokenizer = Tokenizer(
    num_words=MAX_WORDS,
    oov_token="<OOV>"
)

tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_val_seq = tokenizer.texts_to_sequences(X_val)
X_test_seq = tokenizer.texts_to_sequences(X_test)

X_train_pad = pad_sequences(
    X_train_seq,
    maxlen=MAX_LEN,
    padding="post",
    truncating="post",
    dtype="int32"
)

X_val_pad = pad_sequences(
    X_val_seq,
    maxlen=MAX_LEN,
    padding="post",
    truncating="post",
    dtype="int32"
)

X_test_pad = pad_sequences(
    X_test_seq,
    maxlen=MAX_LEN,
    padding="post",
    truncating="post",
    dtype="int32"
)

vocab_size = min(
    MAX_WORDS,
    len(tokenizer.word_index) + 1
)

print("Tokenizer vocabulary size:", vocab_size)
print("Training sequence shape:", X_train_pad.shape)

# ======================================================
# 7. Build the pretrained embedding matrix
# ======================================================
embedding_matrix = np.zeros(
    (vocab_size, embedding_dim),
    dtype=np.float32
)

matched_words = 0

for word, idx in tokenizer.word_index.items():
    if idx >= vocab_size:
        continue

    if word in word_vectors.key_to_index:
        embedding_matrix[idx] = word_vectors[word]
        matched_words += 1

total_words = max(1, vocab_size - 1)

print(
    f"Word2Vec coverage: {matched_words}/{total_words} "
    f"({100 * matched_words / total_words:.2f}%)"
)

# ======================================================
# 8. Build the BiLSTM classifier
# ======================================================
model = Sequential([
    Embedding(
        input_dim=vocab_size,
        output_dim=embedding_dim,
        weights=[embedding_matrix],
        trainable=EMBEDDING_TRAINABLE,
        mask_zero=True,
        name="pretrained_word2vec"
    ),

    Bidirectional(
        LSTM(
            128,
            dropout=0.30,
            recurrent_dropout=0.0
        ),
        name="bidirectional_lstm"
    ),

    Dense(64, activation="relu"),
    Dropout(0.30),

    Dense(num_classes, activation="softmax")
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.001
    ),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

# ======================================================
# 9. Train the model
# ======================================================
callbacks = [
    EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True,
        verbose=1
    ),

    ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=2,
        min_lr=1e-6,
        verbose=1
    )
]

history = model.fit(
    X_train_pad,
    np.asarray(y_train),
    validation_data=(X_val_pad, np.asarray(y_val)),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=callbacks,
    verbose=1
)

# ======================================================
# 10. Evaluate on the held-out test set
# ======================================================
test_loss, test_accuracy = model.evaluate(
    X_test_pad,
    np.asarray(y_test),
    verbose=0
)

print(f"\nTest loss: {test_loss:.4f}")
print(f"Test accuracy: {test_accuracy:.4f}")

y_prob = model.predict(X_test_pad, verbose=0)
y_pred = np.argmax(y_prob, axis=1)

print("\nClassification report:")
print(
    classification_report(
        y_test,
        y_pred,
        labels=np.arange(num_classes),
        target_names=label_encoder.classes_,
        digits=4,
        zero_division=0
    )
)

precision_macro, recall_macro, f1_macro, _ = (
    precision_recall_fscore_support(
        y_test,
        y_pred,
        average="macro",
        zero_division=0
    )
)

precision_weighted, recall_weighted, f1_weighted, _ = (
    precision_recall_fscore_support(
        y_test,
        y_pred,
        average="weighted",
        zero_division=0
    )
)

print(f"Accuracy:        {accuracy_score(y_test, y_pred):.4f}")
print(f"Macro Precision: {precision_macro:.4f}")
print(f"Macro Recall:    {recall_macro:.4f}")
print(f"Macro F1:        {f1_macro:.4f}")
print(f"Weighted F1:     {f1_weighted:.4f}")

# ======================================================
# 11. Plot accuracy and loss
# ======================================================
plt.figure(figsize=(8, 5))
plt.plot(history.history["accuracy"], label="Training accuracy")
plt.plot(history.history["val_accuracy"], label="Validation accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("BiLSTM + Word2Vec Accuracy")
plt.legend()
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(history.history["loss"], label="Training loss")
plt.plot(history.history["val_loss"], label="Validation loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("BiLSTM + Word2Vec Loss")
plt.legend()
plt.tight_layout()
plt.show()
#====================================
# Convert true labels to one-hot/binary format
y_test_bin = label_binarize(
    y_test,
    classes=np.arange(num_classes)
)

# Calculate ROC-AUC scores
macro_auc = roc_auc_score(
    y_test,
    y_prob,
    multi_class="ovr",
    average="macro"
)

weighted_auc = roc_auc_score(
    y_test,
    y_prob,
    multi_class="ovr",
    average="weighted"
)

print(f"Macro ROC-AUC:    {macro_auc:.4f}")
print(f"Weighted ROC-AUC: {weighted_auc:.4f}")

# Plot one-vs-rest ROC curves
plt.figure(figsize=(9, 7))

for i, class_name in enumerate(label_encoder.classes_):
    fpr, tpr, _ = roc_curve(
        y_test_bin[:, i],
        y_prob[:, i]
    )

    class_auc = auc(fpr, tpr)

    plt.plot(
        fpr,
        tpr,
        linewidth=2,
        label=f"{class_name} (AUC = {class_auc:.3f})"
    )

plt.plot([0, 1], [0, 1], linestyle="--", label="Random classifier")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Multiclass ROC Curves: CNN + Word2Vec")
plt.legend(loc="lower right")
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()
#++++++++++++++++++++++++++++
# ======================================================
# 12. Confusion matrix
# ======================================================
cm = confusion_matrix(
    y_test,
    y_pred,
    labels=np.arange(num_classes)
)

plt.figure(figsize=(8, 6))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=label_encoder.classes_,
    yticklabels=label_encoder.classes_
)
plt.xlabel("Predicted category")
plt.ylabel("Actual category")
plt.title("BiLSTM + Word2Vec Confusion Matrix")
plt.tight_layout()
plt.show()

# ======================================================
# 13. Predict a new Amharic question
# ======================================================
def predict_question_type(question):
    seq = tokenizer.texts_to_sequences([str(question)])

    padded = pad_sequences(
        seq,
        maxlen=MAX_LEN,
        padding="post",
        truncating="post",
        dtype="int32"
    )

    probabilities = model.predict(padded, verbose=0)[0]
    pred_idx = int(np.argmax(probabilities))

    return {
        "question": question,
        "predicted_category": label_encoder.inverse_transform(
            [pred_idx]
        )[0],
        "confidence": float(probabilities[pred_idx]),
        "class_probabilities": {
            label_encoder.classes_[i]: float(probabilities[i])
            for i in range(num_classes)
        }
    }

example_question = "ወባ ምንድን ነው?"
print("\nExample prediction:")
print(predict_question_type(example_question))

# ======================================================
# 14. Save model and preprocessing objects
# ======================================================
model.save("amharic_bilstm_word2vec.keras")

with open("amharic_bilstm_tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)

with open("amharic_bilstm_label_encoder.pkl", "wb") as f:
    pickle.dump(label_encoder, f)

print("\nModel and preprocessing objects saved.")
